# MiniVLA Benchmark on a Colab GPU (T4 / L4)

Runs the full benchmark for **SmolVLA** on real **LIBERO** frames and writes everything to `results/`:

1. installs the repo with LeRobot 0.6.1,
2. samples a fixed set of 500 LIBERO frames (seed 0, one episode per task, all 40 tasks) from `HuggingFaceVLA/libero`,
3. GPU: FP32 baseline vs fp16 / bf16 / structured pruning on the **same GPU**,
4. CPU: FP32 baseline vs int8 dynamic quantization / pruning on the **same CPU** (on a 50-frame subset, since SmolVLA takes seconds per sample on 2 vCPUs),
5. writes `results/*.json`, `results/RESULTS.md` and one plot per JSON.

**Runtime:** Runtime -> Change runtime type -> T4 GPU (or L4). On a T4, bf16 is reported as *skipped* because the T4 has no native bf16.

**Do not import torch before the install cell**: LeRobot 0.6.1 pins `torch<2.12`, and pip may replace Colab's torch.

In [ ]:
!nvidia-smi
!python --version

## 1. Get the code and install

In [ ]:
import os
if not os.path.exists("minivla-benchmark"):
    !git clone https://github.com/LasyaRamachandruni/minivla-benchmark.git
%cd minivla-benchmark
!git log -1 --oneline

In [ ]:
!pip install -q "lerobot[smolvla,dataset]==0.6.1" "ray[serve,data]>=2.9" bitsandbytes pytest
!pip install -q -e . --no-deps
!python -c "import torch, lerobot; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| lerobot', lerobot.__version__)"

If the line above shows `cuda False` or an import error, use *Runtime -> Restart session* once and re-run from the `%cd` cell (skip the clone).

Sanity check: unit tests plus the integration tests that run LeRobot's real SmolVLA / dataset code on tiny local fixtures (about a minute).

In [ ]:
!python -m pytest -q

## 2. Sample the evaluation frames (downloads only the 40 sampled episodes)

In [ ]:
!python cli.py sample-libero --dataset HuggingFaceVLA/libero --num-frames 500 --episodes-per-task 1 --seed 0 --horizon 10 --cache data_cache/libero_frames.npz

## 3. GPU: SmolVLA fine-tuned on LIBERO (`HuggingFaceVLA/smolvla_libero`)

FP32 baseline, then fp16, bf16, structured MLP pruning (20% of hidden channels, no fine-tuning) and pruning + fp16. Every variant: same frames, same flow-matching noise, 10 warmup + 100 timed runs at batch 1.

In [ ]:
!python cli.py optimize --model smolvla_libero --device cuda --frames-cache data_cache/libero_frames.npz --num-frames 500 --eval-batch-size 16 --N 100 --warmup 10

Optional: bitsandbytes int8 (MLP / vision Linear layers in int8, rest of the backbone fp16). Experimental: it has not been run before this notebook.

In [ ]:
!python cli.py optimize --model smolvla_libero --device cuda --variants fp32,bnb_int8 --frames-cache data_cache/libero_frames.npz --num-frames 500 --eval-batch-size 16 --N 100 --warmup 10 --out results/smolvla_libero_gpu_bnb_int8.json

## 4. GPU: pretrained `lerobot/smolvla_base`

The base model was trained on SO-100 data (6-D joint actions, 3 cameras), not LIBERO. The suite therefore reports only latency and agreement with its own FP32 outputs, never error vs LIBERO ground truth.

In [ ]:
!python cli.py optimize --model smolvla_base --device cuda --frames-cache data_cache/libero_frames.npz --num-frames 500 --eval-batch-size 16 --N 100 --warmup 10

## 5. CPU: FP32 vs int8 dynamic quantization vs pruning

Same frames (a seeded 50-frame subset), 5 warmup + 50 timed runs per variant. Slow on Colab's 2 vCPUs (expect tens of minutes); skip it if you only need GPU numbers.

In [ ]:
!python cli.py optimize --model smolvla_libero --device cpu --frames-cache data_cache/libero_frames.npz --num-frames 50 --eval-batch-size 1 --N 50 --warmup 5

## 6. Table and plots

In [ ]:
!python cli.py compare --results "results/*.json" --markdown results/RESULTS.md

In [ ]:
import glob
from IPython.display import Image, Markdown, display
display(Markdown(open("results/RESULTS.md").read()))
for png in sorted(glob.glob("results/*.png")):
    display(Image(png))

## 7. Save the results

Download `results.zip`, unzip it into the repo's `results/` folder on your machine, then commit and push.

In [ ]:
!zip -r results.zip results
from google.colab import files
files.download("results.zip")